# 11. シナリオ・プランニング（Scenario Planning） — 練習問題

**対象技術**: 量子コンピューティング

シナリオ・プランニングは、技術を取り巻く環境の不確実性を「影響度」と「不確実性」の二軸で評価し、両方が高い臨界不確実性のうち上位2個を軸に選んで2×2マトリクスを作る手法である。4つのシナリオ（未来文脈）を生成し、各シナリオ下でインパクト指標を評価して、複数の戦略案から全シナリオで頑健な戦略を maximin 基準で選ぶ。このノートブックでは量子コンピューティングの量子安全移行戦略を題材に、その一連の流れを段階的に実行する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 駆動力とインパクト指標の定義

量子コンピューティングの駆動力を、影響度・不確実性スコア（1=低〜5=高）と低極・高極の名前を付けて定義する。両方が高い駆動力が「臨界不確実性」である。あわせて評価するインパクト指標と、その重み、戦略案も定義する。

In [ ]:
# 各駆動力に (名称, 影響度, 不確実性, 低極の名前, 高極の名前) を与える。
DRIVERS = [
    ("量子技術の進歩速度",   5, 5, "停滞", "急進"),
    ("国際秩序の方向",       5, 5, "分断", "協調"),
    ("量子人材の供給",       4, 3, "枯渇", "潤沢"),
    ("公的研究投資の規模",   3, 2, "縮小", "拡大"),
    ("社会の量子リテラシー", 2, 3, "低位", "高位"),
    ("PQC標準化の進捗",      4, 2, "遅延", "順調"),
]

# 評価したいインパクトの軸と重み(評価者の価値判断)。
IMPACT_NAMES = ["暗号安全保障", "産業競争力", "社会的公平性"]
IMPACT_WEIGHTS = np.array([0.4, 0.35, 0.25])

# 戦略案。
STRATEGIES = [
    "即時かつ全面的なPQC移行",
    "段階的・選択的な移行",
    "静観(現状維持)",
]

## 1. 軸の自動選定

影響度×不確実性の積（臨界度）が大きい上位2駆動力を、2×2マトリクスの軸として自動選定する。

In [ ]:
def select_axes(drivers):
    """影響度×不確実性の積が大きい上位2駆動力を軸として選ぶ。"""
    scored = []
    for idx, (name, impact, uncert, lo, hi) in enumerate(drivers):
        criticality = impact * uncert  # 臨界度
        scored.append((criticality, idx, name, lo, hi))
    scored.sort(reverse=True)  # 臨界度の降順
    return scored[0], scored[1], scored


axis1, axis2, scored = select_axes(DRIVERS)
print("[駆動力の臨界度評価] 臨界度 = 影響度 × 不確実性")
print("-" * 70)
for crit, idx, name, lo, hi in scored:
    mark = " <= 軸に採用" if (crit, idx, name, lo, hi) in (axis1, axis2) else ""
    print(f"  {name:<18} 臨界度={crit:>2}{mark}")

## 2. シナリオ生成

選定した2軸の両極を直交させて、4つのシナリオを構成する。

In [ ]:
def build_scenarios(axis1, axis2):
    """2軸の両極を直交させて4シナリオを構成する。"""
    _, _, name1, lo1, hi1 = axis1
    _, _, name2, lo2, hi2 = axis2
    scenarios = [
        (hi1, lo2, "量子覇権競争"),       # 進歩・急進 × 分断
        (hi1, hi2, "協調的量子社会"),     # 進歩・急進 × 協調
        (lo1, lo2, "長い停滞"),           # 進歩・停滞 × 分断
        (lo1, hi2, "分断下のNISQ実用化"), # 進歩・停滞 × 協調寄りの実利
    ]
    return scenarios, (name1, name2)


scenarios, axis_names = build_scenarios(axis1, axis2)
print(f"[選定された2軸] 軸1='{axis_names[0]}'  軸2='{axis_names[1]}'")
print("\n[2×2マトリクスから生成された4シナリオ]")
print("-" * 70)
for i, (p1, p2, sname) in enumerate(scenarios):
    print(f"  S{i+1}: 「{sname}」  ({axis_names[0]}={p1} / {axis_names[1]}={p2})")

## 3. ペイオフ評価

各シナリオ下で戦略案ごとに、複数インパクト指標の評価値（-5〜+5）を専門家評価を模した想定値として定義し、インパクト重みで加重してシナリオ別の単一ペイオフに集約する。

In [ ]:
def payoff_matrix():
    """戦略 × シナリオ のペイオフ行列(各セルはインパクト指標ベクトル)。
    シナリオ列順: 覇権競争, 協調社会, 長い停滞, NISQ実用化
    """
    s0 = np.array([          # 戦略0: 即時全面PQC移行
        [ 4,  3, -2,  3],    # 暗号安全保障
        [ 1,  2, -1,  1],    # 産業競争力
        [ 2,  3,  0,  2],    # 社会的公平性
    ])
    s1 = np.array([          # 戦略1: 段階的・選択的移行
        [ 2,  3,  1,  3],
        [ 2,  3,  2,  3],
        [ 1,  3,  2,  2],
    ])
    s2 = np.array([          # 戦略2: 静観
        [-5, -1,  4,  0],
        [-3,  0,  3,  1],
        [-2,  0,  2,  0],
    ])
    return [s0, s1, s2]


def weighted_payoff(impact_matrix, weights):
    """インパクト指標 × シナリオ を重み加重しシナリオ別ペイオフに集約する。"""
    return weights @ impact_matrix


matrices = payoff_matrix()
print(f"[インパクト指標] {IMPACT_NAMES}  重み={list(IMPACT_WEIGHTS)}")
print("\n[戦略 × シナリオ ペイオフ表] (重み加重後のスコア)")
print("-" * 70)
header = "  戦略\\シナリオ".ljust(28) + "".join(f"{s[2][:8]:>11}" for s in scenarios)
print(header)

strategy_payoffs = []
for si, strat in enumerate(STRATEGIES):
    wp = weighted_payoff(matrices[si], IMPACT_WEIGHTS)
    strategy_payoffs.append(wp)
    row = f"  {strat:<24}" + "".join(f"{v:>11.2f}" for v in wp)
    print(row)
strategy_payoffs = np.array(strategy_payoffs)  # (戦略, シナリオ)

## 4. maximin 基準による頑健戦略のランキング

各戦略の全シナリオ最小ペイオフ（最悪ケース）を求め、それが高い順に並べる。最悪ケースの底上げを評価するのが maximin 基準である。

In [ ]:
worst_case = strategy_payoffs.min(axis=1)  # 戦略ごとの最悪値
order = np.argsort(-worst_case)

print("[maximin 基準による頑健戦略ランキング]")
print("  (各戦略の全シナリオ最小ペイオフ = 最悪ケース。これが高いほど頑健)")
print("-" * 70)
for rank, si in enumerate(order, start=1):
    worst_scen = scenarios[int(np.argmin(strategy_payoffs[si]))][2]
    print(f"  {rank}位: {STRATEGIES[si]:<24} "
          f"最悪ペイオフ={worst_case[si]:>6.2f} (最悪シナリオ:「{worst_scen}」)")

best = order[0]
print("\n[解釈]")
print(f"  maximin 基準で最も頑健な戦略は『{STRATEGIES[best]}』。")
print("  どのシナリオが実現しても壊滅的な結果を避けられる点が選定理由。")
print("  特定シナリオでの最大便益ではなく、最悪ケースの底上げを評価している。")

## 可視化1: 駆動力の影響度×不確実性マップ

駆動力を「影響度×不確実性」の散布図に配置し、臨界度が最大の上位2駆動力（＝主軸）を強調する。右上に位置する駆動力ほどシナリオの軸にふさわしい。

In [ ]:
axis_idx = {axis1[1], axis2[1]}

fig, ax = plt.subplots(figsize=(7, 6))
for idx, (name, impact, uncert, lo, hi) in enumerate(DRIVERS):
    is_axis = idx in axis_idx
    ax.scatter(impact, uncert, s=420 if is_axis else 240,
               c="#c0504d" if is_axis else "#7fb8d6",
               edgecolors="#1f3f5c", linewidths=1.5, zorder=3)
    ax.annotate(f"D{idx+1}", (impact, uncert), ha="center", va="center",
                fontsize=9, fontweight="bold", zorder=4)

ax.axhline(3.5, color="#999999", linestyle="--", linewidth=1)
ax.axvline(3.5, color="#999999", linestyle="--", linewidth=1)
ax.text(4.8, 4.8, "critical\nuncertainties", ha="right", va="top",
        fontsize=9, color="#c0504d")
ax.set_xlabel("Impact")
ax.set_ylabel("Uncertainty")
ax.set_title("Driving forces: Impact x Uncertainty (axes highlighted)")
ax.set_xlim(0.5, 5.7)
ax.set_ylim(0.5, 5.7)
legend = "\n".join(f"D{i+1}: {d[0]}" for i, d in enumerate(DRIVERS))
ax.text(1.05, 0.98, legend, transform=ax.transAxes, fontsize=8,
        va="top", ha="left",
        bbox=dict(boxstyle="round", fc="#f3f6f8", ec="#cccccc"))
fig.tight_layout()
plt.show()

## 可視化2: 戦略×シナリオのペイオフ・ヒートマップ

戦略×シナリオの加重ペイオフをヒートマップで示す。各戦略行の最小値（最悪ケース）に枠を付け、maximin で選ばれた頑健戦略を注記する。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(strategy_payoffs, cmap="RdYlGn", aspect="auto",
               vmin=-3, vmax=3)
scen_labels = [f"S{i+1}" for i in range(len(scenarios))]
strat_labels = [f"ST{i+1}" for i in range(len(STRATEGIES))]
ax.set_xticks(range(len(scenarios)))
ax.set_xticklabels(scen_labels)
ax.set_yticks(range(len(STRATEGIES)))
ax.set_yticklabels(strat_labels)

for si in range(len(STRATEGIES)):
    worst_j = int(np.argmin(strategy_payoffs[si]))
    for sj in range(len(scenarios)):
        ax.text(sj, si, f"{strategy_payoffs[si, sj]:.2f}",
                ha="center", va="center", fontsize=9)
    ax.add_patch(plt.Rectangle((worst_j - 0.5, si - 0.5), 1, 1,
                 fill=False, edgecolor="#1f3f5c", linewidth=3))

ax.set_title(f"Strategy x Scenario payoff  (maximin pick: ST{best+1})")
ax.set_xlabel("Scenario  " + " / ".join(
    f"S{i+1}={s[2]}" for i, s in enumerate(scenarios)))
fig.colorbar(im, ax=ax, label="weighted payoff")
fig.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザイン論文においてシナリオ・プランニングは、しばしば論文の構成そのものを規定する骨格として用いられる。臨界不確実性を二軸に立て、四つの対照的な未来文脈を設定し、各文脈の下で評価対象のインパクトを評価し直す——この章立てが採られた時点で、論文は「未来はこうなる」という単線の予測を方法論的に放棄している。そこで生まれる結論は確率の言明ではなく、頑健性の主張である。すなわち「いずれの文脈でも一定の便益を生む戦略はこれである」「特定の文脈に賭けた戦略はこの文脈で破綻する」という、条件付きの判断地図が成果物となる。

この手法が結論に持ち込む第一の規定力は時間観にある。シナリオ・プランニングは未来を過去の延長としてではなく、複数あって確定しないものとして扱う。ゆえに論文は「最も起こりやすい未来への最適化」を語らず、不確実性そのものを正面に据える多元主義へと方向づけられる。第二は境界設定である。世界の不確実性は本来多次元だが、二軸への単純化を経た瞬間、軸に選ばれなかった駆動力は四つの文脈を貫く背景に退き、結論の主役から構造的に排除される。

価値の所在という点では、シナリオ・プランニングは戦略評価の基準——maximin か minimax-regret か期待値か——に価値判断を埋め込む。とりわけ「最悪値を最良にする」基準を採れば、論文の結論はリスク回避的なヘッジ戦略へと体系的に傾く。どの未来でも壊滅を避ける選択肢が頑健と呼ばれ、特定の好機に賭ける積極策は退けられやすい。したがってこの手法を用いた論文は、単一予測を拒む知的誠実さと引き換えに、選ばれなかった軸の捨象とヘッジ志向という二つの偏りを結論に抱え込む。

## 発展課題

**課題A**: maximin の代わりに minimax-regret（最大後悔の最小化）基準を実装せよ。各シナリオでの「最良戦略のペイオフ - その戦略のペイオフ」を後悔とし、戦略ごとの最大後悔が最小の戦略を選ぶ。maximin の選択とどう違うかを論ぜよ。

**課題B**: `DRIVERS` のスコアに ±1 の摂動を一様乱数で加える試行を多数回行い、「どの2駆動力が軸に選ばれるか」がどれだけ安定かを集計せよ。軸選定が不安定な場合、シナリオ設計のどこに注意すべきかを考察せよ。